<a href="https://colab.research.google.com/github/Swag-Pseudopy/Escaping-Preference-Collapse/blob/main/stochasticity_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
stochasticity_experiment.py
---------------------------
Validates Lemma 5 (Stochasticity Gap, Eq. 8 in the paper).

Compares three conditions of Nash-MD under mini-batch noise:
  (a) No noise (reference)
  (b) Noise + uncorrected tau  -> diverges
  (c) Noise + corrected tau*   -> sustains stable orbit

Outputs:  figures/fig_stochastic.pdf
          figures/fig_stochastic_entropy.pdf

Run time: < 90 seconds on a standard CPU.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

# ── reproducibility ──────────────────────────────────────────────────────────
RNG = np.random.default_rng(42)
os.makedirs("figures", exist_ok=True)

# ── helpers ──────────────────────────────────────────────────────────────────

def make_condorcet(n: int) -> np.ndarray:
    """Symmetric n-action Condorcet cycle: A[i, (i+1) % n] = 1."""
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = 1.0
        A[(i + 1) % n, i] = -1.0
    return A


def excess_energy(p: np.ndarray, n: int) -> float:
    """C_tilde = -sum log p_i - n log n."""
    Cmin = n * np.log(n)
    return -np.sum(np.log(np.clip(p, 1e-15, None))) - Cmin


def eexp(p: np.ndarray, A: np.ndarray) -> float:
    """E_exp(p) = (k/2) * sum_j p_j * (Ap)_j^2."""
    k = len(p)
    Ap = A @ p
    return (k / 2.0) * np.dot(p, Ap ** 2)


def nash_md_step(
    p: np.ndarray,
    A: np.ndarray,
    eta: float,
    tau: float,
    p_ref: np.ndarray,
    noise_std: float = 0.0,
    rng: np.random.Generator = None,
) -> np.ndarray:
    """
    One step of regularized Nash-MD (Eq. 4 in the paper).

    p_{t+1,i} ∝ p_{t,i}^{1-eta*tau} * p_ref_i^{eta*tau} * exp(eta*(A_hat @ p)_i)

    A_hat = A + xi  where xi is skew-symmetric noise with entries N(0, noise_std^2).
    """
    n = len(p)
    if noise_std > 0.0 and rng is not None:
        raw = rng.normal(0.0, noise_std, (n, n))
        xi = (raw - raw.T) / 2.0          # enforce skew-symmetry
        A_hat = A + xi
    else:
        A_hat = A

    Ap = A_hat @ p
    log_p_new = (1.0 - eta * tau) * np.log(np.clip(p, 1e-15, None)) \
                + eta * tau * np.log(np.clip(p_ref, 1e-15, None)) \
                + eta * Ap
    # subtract log-sum-exp for numerical stability
    log_p_new -= log_p_new.max()
    p_new = np.exp(log_p_new)
    return p_new / p_new.sum()


def run_experiment(
    n: int,
    eta: float,
    tau: float,
    noise_std: float,
    n_iters: int,
    p0: np.ndarray,
    adaptive_tau: bool,
    rng: np.random.Generator,
) -> tuple:
    """Run Nash-MD and return (energies, entropies)."""
    A = make_condorcet(n)
    p_ref = np.ones(n) / n
    p = p0.copy()
    energies = []
    entropies = []

    for _ in range(n_iters):
        energies.append(excess_energy(p, n))
        entropies.append(-np.dot(p, np.log(np.clip(p, 1e-15, None))))

        if adaptive_tau and noise_std > 0.0:
            # corrected tau*: Eq. 9 in the paper
            # Tr(Sigma) for skew-sym noise: n*(n-1)/2 pairs each variance 2*sigma^2
            # Expected contribution: (eta^2/2) * sum_j p_j * Sigma_jj
            # For isotropic noise xi_ij ~ N(0, s^2), (Ap)_j variance = sum_i A_ij^2 * s^2
            Ap_var_per_j = np.array([(A[j] ** 2).sum() * noise_std ** 2 for j in range(n)])
            tr_sigma = np.dot(p, Ap_var_per_j)
            e_exp = eexp(p, A)
            if e_exp > 1e-10:
                correction = 1.0 + tr_sigma / (2.0 * e_exp)
            else:
                correction = 1.0
            tau_eff = tau * correction
        else:
            tau_eff = tau

        p = nash_md_step(p, A, eta, tau_eff, p_ref, noise_std, rng)

    return np.array(energies), np.array(entropies)


# ── experiment parameters ─────────────────────────────────────────────────────
N_ACTIONS   = 3
ETA         = 0.05
TAU_BASE    = ETA          # stable-regime tau (kappa=1)
NOISE_STD   = 0.15         # mini-batch noise level
N_ITERS     = 1500
N_SEEDS     = 20

# fixed starting point (away from Nash eq for visibility)
P0 = np.array([0.7, 0.2, 0.1])

conditions = [
    {"label": "No noise",              "noise": 0.0,        "adaptive": False, "color": "#444441", "ls": "-"},
    {"label": r"Noise, uncorrected $\tau$", "noise": NOISE_STD, "adaptive": False, "color": "#D85A30", "ls": "--"},
    {"label": r"Noise, corrected $\tau^*$", "noise": NOISE_STD, "adaptive": True,  "color": "#1D9E75", "ls": "-"},
]

print("Running stochasticity gap experiment …")

all_results = {}
for cond in conditions:
    energies_list = []
    for seed in range(N_SEEDS):
        rng_local = np.random.default_rng(seed)
        E, H = run_experiment(
            n=N_ACTIONS,
            eta=ETA,
            tau=TAU_BASE,
            noise_std=cond["noise"],
            n_iters=N_ITERS,
            p0=P0,
            adaptive_tau=cond["adaptive"],
            rng=rng_local,
        )
        energies_list.append(E)
    all_results[cond["label"]] = np.stack(energies_list)   # (N_SEEDS, N_ITERS)
    print(f"  {cond['label']:40s} done.")

# ── figure ───────────────────────────────────────────────────────────────────
iters = np.arange(N_ITERS)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax_ts, ax_final = axes

# --- left: time series (mean ± 1 std) ---
for cond in conditions:
    mat = all_results[cond["label"]]
    mu  = mat.mean(axis=0)
    sd  = mat.std(axis=0)
    ax_ts.plot(iters, mu,  color=cond["color"], ls=cond["ls"], lw=1.5, label=cond["label"])
    ax_ts.fill_between(iters, mu - sd, mu + sd, color=cond["color"], alpha=0.15)

ax_ts.set_xlabel("Iteration", fontsize=11)
ax_ts.set_ylabel(r"Excess energy $\tilde{C}_t$", fontsize=11)
ax_ts.set_title("Time-series (mean ± 1 SD, 20 seeds)", fontsize=11)
ax_ts.legend(fontsize=9)
ax_ts.set_xlim(0, N_ITERS)
ax_ts.set_ylim(bottom=0)

# --- right: final steady-state energy across noise levels ---
noise_levels = np.linspace(0.0, 0.30, 12)
final_uncorrected = []
final_corrected   = []

for sigma in noise_levels:
    vals_u, vals_c = [], []
    for seed in range(N_SEEDS):
        rng_local = np.random.default_rng(seed + 100)
        E_u, _ = run_experiment(N_ACTIONS, ETA, TAU_BASE, sigma, N_ITERS, P0, False, rng_local)
        E_c, _ = run_experiment(N_ACTIONS, ETA, TAU_BASE, sigma, N_ITERS, P0, True,  rng_local)
        # take mean of last 200 iterations as steady-state estimate
        vals_u.append(E_u[-200:].mean())
        vals_c.append(E_c[-200:].mean())
    final_uncorrected.append(np.mean(vals_u))
    final_corrected.append(np.mean(vals_c))

ax_final.plot(noise_levels, final_uncorrected, color="#D85A30", marker="o", ms=4,
              label=r"Uncorrected $\tau$")
ax_final.plot(noise_levels, final_corrected,   color="#1D9E75", marker="s", ms=4,
              label=r"Corrected $\tau^*$")
ax_final.axhline(0, color="#444441", lw=0.8, ls=":")
ax_final.set_xlabel(r"Noise level $\sigma$", fontsize=11)
ax_final.set_ylabel(r"Steady-state $\tilde{C}^*$ (mean, last 200 iters)", fontsize=11)
ax_final.set_title(r"Effect of $\tau^*$ correction vs. noise level", fontsize=11)
ax_final.legend(fontsize=9)

fig.tight_layout()
fig.savefig("figures/fig_stochastic.pdf", bbox_inches="tight", dpi=150)
print("Saved figures/fig_stochastic.pdf")
plt.close(fig)

Running stochasticity gap experiment …
  No noise                                 done.
  Noise, uncorrected $\tau$                done.
  Noise, corrected $\tau^*$                done.
Saved figures/fig_stochastic.pdf
